# Monte Carlo Delta Estimation

This notebook explores Monte Carlo methods for estimating the sensitivity (delta) of option values with respect to the initial stock price. We consider:

- Finite-difference (bump-and-revalue) estimators for a European call option and for a digital-style payoff.
- Pathwise (trajectory-based) estimators for a smoothed digital payoff.
- Likelihood ratio (score function) estimators for the same smoothed digital payoff.

Throughout, we work under a geometric Brownian motion model with constant parameters and focus on how bias, variance, and payoff smoothness influence the behaviour of delta estimators.

**Structure**

1. European call delta via bump-and-revalue (finite differences) and MSE-based bump selection.
2. Digital option delta with and without smoothing.
3. Smoothed digital payoff with pathwise and likelihood-ratio delta estimators and a comparative discussion.

##  1. European Options: Approximation of $\Delta$ using Bump-and-Revalue (Forward Finite Differences) method

The **delta** of an option measures how sensitive the option value is to changes in the initial stock price $S_0$.

Delta is defined as:

$$\Delta = \frac{\partial V}{\partial S_0}$$

Under risk-neutral pricing, the option value is:

$$V(S_0) = e^{-rT} \mathbb{E}[\text{payoff}(S_T)]$$

where $S_T$ depends on both $S_0$ and the random normal variables driving the simulation.


From basic calculus:

$$f'(x_0) \approx \hat{d}f_x(x_0; h) = \frac{f(x_0 + h) - f(x_0)}{h}$$

Applying this idea to the option value:

$$\Delta \approx \frac{V(S_0 + h) - V(S_0)}{h}$$

Because the terminal stock price depends on randomness, we simulate:

$$S_T = S_T(S_0, Z)$$

where $Z$ represents the random normal variables. The Monte Carlo delta estimator becomes:

$$\hat{\Delta}(h) = \frac{V(S_T(S_0 + h, Z)) - V(S_T(S_0, Z))}{h}$$

In [ ]:
# Imports and model parameters
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import norm
import pandas as pd
from mpl_toolkits.mplot3d import Axes3D

# Global model parameters (can be adjusted)
T = 1            # time to maturity (1 year)
N = 252          # number of time steps for Euler discretization (trading days)
M = 5000         # number of Monte Carlo paths
S0 = 100         # initial stock price
K = 99           # strike price
r = 0.06         # risk-free interest rate
sigma = 0.2      # sigmaatility of the stock 
H = 100          # number of h values to test (grid size)
h_values = np.linspace(0.001, 0.3, H)

So, we estimate the delta sensitivity defined as

$$
\Delta_0 = \frac{\partial V}{\partial S_0},
$$

where $S_0$ is the initial stock price using the previously mentioned forward finite differences approximation. 

We simulate first the stock path using the Euler discretization:

$$
S_n = S_{n-1} \left( 1 + r\,dt + \sigma \sqrt{dt}\, Z_n \right),
$$

where:

- $r$ is the risk-free rate,
- $\sigma$ is the sigmaatility,
- $dt = T/N$,
- $Z_n \sim \mathcal{N}(0,1)$.

In [ ]:
# Euler simulation
def euler_forward(S0, normals, dt):

    M,N_steps = normals.shape
    S = np.zeros((M,N_steps+1))
    S[:,0] = S0

    for n in range(N_steps):
        S[:,n+1] = S[:,n] + S[:,n]*(r*dt + sigma*np.sqrt(dt)*normals[:,n])

    return S

# European call option price
def option_price(S0, normals):
    dt = T/N
    paths = euler_forward(S0, normals, dt)
    ST = paths[:,-1]
    payoff = np.maximum(ST-K, 0)

    return np.exp(-r*T)*payoff

# Black-Scholes delta
def bs_delta(S0, K, r, T, sigma):

    d1 = (np.log(S0/K)+(r+0.5*sigma**2)*T)/(sigma*np.sqrt(T))

    return norm.cdf(d1)

# Actual value of delta
delta_true = bs_delta(S0, K, r, T, sigma)

Let $(Z_{k,j})_{1 \le k \le M,\ 1 \le j \le 2N}$ be standard normal variables.

The estimator is

$$
\widehat{\Delta}(h; M)
=
\frac{1}{Mh}
\sum_{k=1}^{M}
\left[
V\big(S_T(S_0 + h, Z_{k,1}, \dots, Z_{k,N})\big)
-
V\big(S_T(S_0, Z_{k,N+1}, \dots, Z_{k,2N})\big)
\right].
$$

Since

$$
\mathbb{E}[\max(S_T - K, 0)] < \infty,
$$

we may apply the Strong Law of Large Numbers:

$$
\widehat{\Delta}(h; M)
\xrightarrow[M \to \infty]{}
\mathbb{E}
\left[
\frac{
V(S_T(S_0 + h, Z_1, \dots, Z_N))
-
V(S_T(S_0, Z_{N+1}, \dots, Z_{2N}))
}{h}
\right]
=
\mathbb{E}[\widehat{\Delta}_0(h)].
$$

And, then, to approximate $\Delta_0$:

1. Simulate $M \times 2N$ standard normal variables.
2. For each simulation:
   - Use the first $N$ normals to build the path with bumped initial value $S_0 + h$.
   - Use the next $N$ normals to build the path with initial value $S_0$.
3. Compute the payoff difference divided by $h$.
4. Average over $M$ simulations.

We will compare two cases:

### 1) Different Seeds (Independent $Z$)

- The bumped path and unbumped path use independent normal variables.
- Higher variance.
- No correlation benefit.

### 2) Same Seeds (Common Random Numbers)

- The same $Z_1, \dots, Z_N$ are used for both $S_0 + h$ and $S_0$.
- This induces strong positive correlation.
- Variance is significantly reduced.

In [ ]:
# Forward finite difference estimator
def delta_estimator(h, same_seed):

    Z = np.random.randn(M,2*N)

    if same_seed:

        Z_common = Z[:,:N]
        V_plus = option_price(S0+h, Z_common)
        V_0 = option_price(S0, Z_common)

    else:

        Z_plus = Z[:,:N]
        Z_0 = Z[:,N:]
        V_plus = option_price(S0+h, Z_plus)
        V_0 = option_price(S0, Z_0)

    diffs = (V_plus - V_0)/h

    estimator = np.mean(diffs)
    variance = np.var(diffs,ddof=1)/M
    bias = estimator - delta_true
    mse = bias**2 + variance

    return estimator, bias, variance, mse

### Bias Vs Variance of the Finite Differences Method

By a Taylor approximation around $x_0$ we find the bias to be given by

$$\hat{d}f_x(x_0; h) - \frac{df(x_0)}{dx} = \frac{1}{h} \left( f(x_0 + h) - f(x_0) - hf'(x_0) \right)$$

$$= \frac{1}{h} \left( f(x_0) + f'(x_0)h + \frac{1}{2}f''(x_0)h^2 + \mathcal{O}(h^3) - f(x_0) - hf'(x_0) \right)$$

$$= \frac{1}{h} \left( \frac{1}{2}f''(x_0)h^2 + \mathcal{O}(h^3) \right)$$

$$= \mathcal{O}(h).$$


Considering now the variance of the finite difference estimator we end up having approximately:

$$\text{Var}(\hat{d}f_x(x_0; h)) = \frac{\text{Var}(f(x_0 + h)) + \text{Var}(f(x_0)) - 2\text{Cov}(f(x_0 + h), f(x_0))}{h^2}$$

Combining these effects gives us the classical **bias–variance tradeoff**:

- **Small $h$** → small bias but potentially large variance
- **Large $h$** → larger bias but smaller variance

The optimal choice of $h$ balances these two sources of error.

In [ ]:
# Storage arrays
delta_ind = np.zeros(H)
bias_ind = np.zeros(H)
var_ind = np.zeros(H)
mse_ind = np.zeros(H)

delta_corr = np.zeros(H)
bias_corr = np.zeros(H)
var_corr = np.zeros(H)
mse_corr = np.zeros(H)

print("Processing simulations...")
for i, h in enumerate(h_values):
    delta_ind[i], bias_ind[i], var_ind[i], mse_ind[i] = delta_estimator(h, same_seed=0)
    delta_corr[i], bias_corr[i], var_corr[i], mse_corr[i] = delta_estimator(h, same_seed=1)

print("Done.")

# Plot results
plt.figure(figsize=(10,6))
plt.plot(h_values, mse_ind, label="Independent seeds")
plt.plot(h_values, mse_corr, label="Same seeds (common random numbers)")
plt.xlabel("h")
plt.ylabel("MSE (log scale)")
plt.title("MSE of Delta MC (Bump-and-revalue) Estimator")
plt.yscale('log')          
plt.legend()
plt.show()

From the above variance formula for the bump-and-revalue estimator of $\Delta$ we have:

$$
\mathrm{Var}[\hat{\Delta}] =
\mathrm{Var}\left[\frac{V(S_0+h)-V(S_0)}{h}\right]
=
\frac{1}{h^2}
\left(
\mathrm{Var}[V(S_0+h)]
+
\mathrm{Var}[V(S_0)]
-
2\,\mathrm{Cov}[V(S_0+h), V(S_0)]
\right)
$$

With different seeds,

$$
\mathrm{Cov}[V(S_0+h), V(S_0)] = 0 .
$$

However, if the same seed is used,

$$
\mathrm{Cov}[V(S_0+h), V(S_0)] > 0 .
$$

Hence, using the same seed reduces the standard error of the estimator as it is obvious from the plot as well.

In [ ]:
#stats
results_df = pd.DataFrame({
    'h': h_values,
    'delta_ind': delta_ind,
    'bias_ind': bias_ind,
    'var_ind': var_ind,
    'mse_ind': mse_ind,
    'delta_corr': delta_corr,
    'bias_corr': bias_corr,
    'var_corr': var_corr,
    'mse_corr': mse_corr,
})

# Find index of smallest absolute bias for each case
idx_ind = np.argmin(np.abs(results_df['bias_ind']))
idx_corr = np.argmin(np.abs(results_df['bias_corr']))

min_bias_ind = results_df.loc[idx_ind, 'bias_ind']
min_bias_corr = results_df.loc[idx_corr, 'bias_corr']

# Compare which one is closer to zero
if abs(min_bias_ind) < abs(min_bias_corr):

    print("Method: Independent seeds")
    print("h:", results_df.loc[idx_ind, 'h'])
    print("Bias:", results_df.loc[idx_ind, 'bias_ind'])
    print("Delta approximation:", results_df.loc[idx_ind, 'delta_ind'])
    print("Variance:", results_df.loc[idx_ind, 'var_ind'])
    h_star = results_df.loc[idx_ind, 'h']


else:

    print("Method: Same random seeds")
    print("h:", results_df.loc[idx_corr, 'h'])
    print("Bias:", results_df.loc[idx_corr, 'bias_corr'])
    print("Delta approximation:", results_df.loc[idx_corr, 'delta_corr'])
    print("Variance:", results_df.loc[idx_corr, 'var_corr'])
    h_star = results_df.loc[idx_corr, 'h']


Since bias cannot be eliminated by increasing the sample size, whereas variance can be reduced arbitrarily by choosing a sufficiently large number of simulations, we select the bump size $h$ that minimizes the bias. 

Notice that for each run you get different optimal $h$. That's because the bias a function of $h$ is mooth but the noise is not, the argmin can jump around.

In [ ]:
def monteCarlo_forward(S0, N, M, h_star, same_seed):

    if same_seed:
        normals = np.random.normal(size=(M, N))                    
        v_plus = option_price(S0 + h_star, normals)
        v_base = option_price(S0, normals)
    else:
        normals_plus = np.random.normal(size=(M, N))
        normals_base = np.random.normal(size=(M, N))
        v_plus = option_price(S0 + h_star, normals_plus)
        v_base = option_price(S0, normals_base)

    Delta = (v_plus - v_base) / h_star
    return np.mean(Delta)

# Grid of M values
M_grid = np.arange(100, 2001, 10)
approx_same = np.zeros(len(M_grid))
approx_ind  = np.zeros(len(M_grid))

for i, m in enumerate(M_grid):
    approx_same[i] = monteCarlo_forward(S0, N, M, h_star, same_seed=True)
    approx_ind[i]  = monteCarlo_forward(S0, N, M, h_star, same_seed=False)

# Plotting
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
ax1.plot(M_grid, approx_same, label='Same seeds')
ax1.axhline(delta_true, color='red', label='Black–Scholes delta')
ax1.set_xlabel('M')
ax1.set_ylabel('Delta estimate')
ax1.set_title('Common random numbers')
ax1.legend()

ax2.plot(M_grid, approx_ind, label='Independent seeds')
ax2.axhline(delta_true, color='red', label='Black–Scholes delta')
ax2.set_xlabel('M')
ax2.set_ylabel('Delta estimate')
ax2.set_title('Independent random numbers')
ax2.legend()

plt.tight_layout()
plt.show()

From the plots, we see that the forward finite-difference delta estimator using **independent random seeds** fluctuates much more around the true Black–Scholes delta than the estimator that uses **common random numbers** (same seeds). 

Given the relatively narrow range of plausible values for the true delta, these large swings for the independent-seed estimator indicate a much higher Monte Carlo variance. 

This makes the common–random–numbers version clearly preferable in practice, since it delivers estimates that stay much closer to the true delta for the same computational cost.

## 2. Digital Options: Approximation of $\Delta$ using Bump-and-Revalue method.

We will show now why this method is not proper for estimating the delta of a digital call option which pays 1 euro if the stock price at expiry is higher than the strike and otherwise nothing, i.e. $V(S_T) = \mathbf{1}_{S_T \geq K}$


Choose $h>0$ to be a very small number. Then the hedge parameter can be approximated as follows 

$$
\hat{\Delta} = \frac{V(S_0 + h) - V(S_0)}{h}
$$

using the MC method to determine $V(S_0 + h)$ and $V(S_0)$. 

In [ ]:
def option_price_digital(S0, normals):

    dt = T/N
    S = euler_forward(S0, normals, dt)
    ST = S[:,-1]

    payoff = np.exp(-r*T) * (ST > K).astype(float)

    return payoff

def digital_delta_true(S0, K, r, sigma, T):

    d2 = (np.log(S0/K) + (r - 0.5*sigma**2)*T) / (sigma*np.sqrt(T))

    delta = np.exp(-r*T) * norm.pdf(d2) / (S0 * sigma * np.sqrt(T))

    return delta

delta_true = digital_delta_true(S0, K, r, sigma, T)
Z = np.random.randn(M, N)   # common random numbers

def delta_estimator_digital(h):

    V_plus = option_price_digital(S0 + h, Z)
    V_0 = option_price_digital(S0, Z)

    diffs = (V_plus - V_0) / h

    estimator = np.mean(diffs)
    variance = np.var(diffs, ddof=1) / M

    bias = estimator - delta_true
    mse = bias**2 + variance

    return estimator, bias, variance, mse

# Sample sizes and bump sizes
sample_sizes = [10000, 20000, 50000, 100000]
h_test_values = [0.01, 0.02, 0.05]

# Store results
results = np.zeros((len(sample_sizes), len(h_test_values)))

print(f"True digital delta: {delta_true:.6f}")

for i, M_size in enumerate(sample_sizes):

    # regenerate Z with the desired MC size
    Z = np.random.randn(M_size, N)
    M = M_size

    for j, h in enumerate(h_test_values):

        est, bias, var, mse = delta_estimator_digital(h)

        relative_error = abs(est - delta_true) / abs(delta_true) * 100

        results[i, j] = relative_error

df = pd.DataFrame(
    results,
    index=sample_sizes,
    columns=[f'h = {h}' for h in h_test_values]
)

df.index.name = 'MC Size'

print("\nResults for different MC sizes and bump sizes")
print(df.round(2).astype(str) + '%')


For a digital option, this method seems to not work well due to the discontinuity of the payoff function. If the shift-size $h$ is very small and the MC is performed with an insufficient number of paths, the value corresponding to the bumped-scenario might change drastically due to one path crossing the discontinuity boundary of the payoff. It is already evident from the results in the table above - not stable and not nearly as good as with a European option. 

In other words, the indicator function is not everywhere differentiable (either 0 or 1), so the estimation of delta at time 0
$$
\hat{\Delta}_0(h; M) = \frac{e^{-rT}}{h} \left[ \hat{V}(S_0 + h) - \hat{V}(S_0) \right]
$$

where 
- $\hat{V}(S_0)$ is the MC estimate of the digital option price at $S_0$ using $M$ paths
-  $\hat{V}(S_0 + h)$ is the MC estimate at $S_0 + h$ 

using same random seeds, each path contribution will equal to either:
- 0 : when both bumped and unbumped stock prices end on the same side of $K$ (both $\geq K$ or both $<K$)
- $+\frac{e^{-rT}}{h}$ : when the bumped stock price is $\geq K$ and the unbumped stock price is $<K$
- $-\frac{e^{-rT}}{h}$ : when the bumped stock price is $<K$ and the unbumped stock price is $\geq K$

That means either a lot of zeros or large spikes (since $h$ is very small) revealing the instability from the discontinuous payoff.

### 2.1. Pathwise Method Smoothing Method

Before applying the actual method to make the estimation more robust, let's make the first the digital option differentiable with a practical trick. To do this, we approximate the payoff using a logistic
function

$$
\tilde V(S_T;\alpha)=\frac{1}{1+\exp(-\alpha(S_T-K))}
$$

where $\alpha$ controls the steepness of the curve. 

In [ ]:
def digital_logistic_payoff(ST, alpha):
    return 1/(1 + np.exp(-alpha*(ST-K)))

ST_grid = np.linspace(95,105,400)

alpha_values = [1,3,5,10,15]

plt.figure(figsize=(8,5))

# True digital payoff
plt.plot(ST_grid, (ST_grid >= K).astype(float),
         color='black', linestyle='--', label="Digital payoff")

# Logistic approximations
for alpha in alpha_values:
    
    plt.plot(ST_grid,
             digital_logistic_payoff(ST_grid, alpha),
             label=f"Logistic curve for alpha = {alpha}")

plt.xlabel("Value of $S_T$")
plt.ylabel("Smoothening of V(S_T) for different alphas")
plt.title("Smooth approximation of digital payoff")

plt.legend()
plt.grid(alpha=0.3)
plt.show()

Small values of $\alpha$ produce a very smooth curve but introduce
bias because the function deviates from the digital payoff.
Large values of $\alpha$ approximate the digital payoff more closely,
but the estimator variance increases. To determine a suitable value of $\alpha$, we compute the empirical
mean squared error
$$
MSE(\alpha)=Var(\hat{\delta}(\alpha))+
\left(E[\hat{\delta}(\alpha)]-\delta_{true}\right)^2
$$

where $\delta_{true}$ is the analytical Black–Scholes value of the digital delta.

In [ ]:
def option_price_smooth(S0, normals, alpha_array):

    dt = T/N
    S = euler_forward(S0, normals, dt)
    ST = S[:,-1]
    
    payoff = np.exp(-r*T) * np.array([digital_logistic_payoff(ST, alpha) for alpha in alpha_array])
    
    return payoff

def delta_estimator_smooth(h, alpha_input):

    if np.isscalar(alpha_input):
        alpha_array = np.array([alpha_input])
        single_result = True
    else:
        alpha_array = alpha_input
        single_result = False
    
    V_plus = option_price_smooth(S0 + h, Z, alpha_array)
    V_0 = option_price_smooth(S0, Z, alpha_array)
    
    diffs = (V_plus - V_0) / h
    
    estimators = np.mean(diffs, axis=1)
    variances = np.var(diffs, axis=1, ddof=1) / M
    biases = estimators - delta_true
    mses = biases**2 + variances
    
    if single_result:
        return estimators[0], biases[0], variances[0], mses[0]
    else:
        return estimators, biases, variances, mses

alpha_values = np.linspace(5, 25, 15)  

results = []

for h in h_values:

    est_array, bias_array, var_array, mse_array = delta_estimator_smooth(h, alpha_values)
    
    for i, alpha in enumerate(alpha_values):
        results.append({
            "h": h,
            "alpha": alpha,
            "estimator": est_array[i],
            "bias": bias_array[i],
            "variance": var_array[i],
            "mse": mse_array[i]
        })

results_df = pd.DataFrame(results)

best_row = results_df.loc[results_df["mse"].idxmin()]

h_star = best_row["h"]
alpha_star = best_row["alpha"]

print("Optimal h:", h_star)
print("Optimal alpha:", alpha_star)
print("Minimum MSE:", best_row["mse"])

The nested loop over $h$ and $\alpha$ performs a systematic grid search. For each combination, we compute the MSE, which balances squared bias (from smoothing) and variance (from Monte Carlo noise) to end up with the optimal pair $(h, \alpha)$ that minimizes this metric. 

Also, the intuition behind all this is the replacement of the Dirac delta at the strike price with a smooth bump. In other words $\alpha$ controls how concentrated the bump $h$ is!

In [ ]:
delta_final, bias_final, var_final, mse_final = delta_estimator_smooth(h_star, alpha_star)

print("Final Delta Estimate:", delta_final)
print("Bias:", bias_final)
print("Variance:", var_final)
print("MSE:", mse_final)

In [ ]:
H_grid, A_grid = np.meshgrid(h_values, alpha_values)
Z = results_df.pivot(index="alpha", columns="h", values="mse").values

fig = plt.figure(figsize=(8,5))
ax = fig.add_subplot(111, projection='3d')

surf = ax.plot_surface(H_grid, A_grid, Z, cmap='viridis')

ax.set_xlabel("h")
ax.set_ylabel("alpha")
ax.set_zlabel("MSE")

cbar = fig.colorbar(surf, shrink=0.5, pad=0.15)  
plt.title("MSE Surface for (h, alpha)")
plt.tight_layout() 
plt.show()

The surface above in terms of MSE, $\alpha$, and $h$ applies that low MSE (near zero) appears along a diagonal band where $h$ and $\alpha$ are "balanced".